**Name: Rudra P Patel**



**Roll No.: 25MCE023**


**Practical-4: Build a language model using RNN. Write functions to sample
novel sentences and find the probability of the input sentence.  Also,
use Recurrent Neural Network for Sentiment Analysis.**

predicts the next word in a sequence.


Jack and Jill went

Jack → and  
and → Jill  
Jill → went  

like:
X = [1, 2, 3]
y = [1, 2, 3, 4]

y represents the target sequence where each position corresponds to the next word the model should predict

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

# Data preparing-
data = """ Jack and Jill went up the hill .\n To fetch a pail of water .\n Jack fell down and broke his crown .\n And Jill came tumbling after ."""
print("Data:", data, type(data))

# returns a list of strings
data_splitted = data.split('\n')
print("Data_Splitted:", data_splitted, type(data_splitted))

# Custom Tokenizer to replicate Keras Tokenizer behavior
class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):
        self.word_index = {}
        self.index_word = {0: 'PAD'}
        self.filters = filters
        self.word_count = {}

    def fit_on_texts(self, texts):
        current_index = 1
        for text in texts:
            # Basic cleaning, remove filters, convert to lowercase
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1
                self.word_count[word] = self.word_count.get(word, 0) + 1


    # here y we need to convert the text to token bez for tensor array
    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()
            seq = [self.word_index[word] for word in words if word in self.word_index]
            sequences.append(seq)
        return sequences

# Instantiate and fit tokenizer
tokenizer_pt = PyTorchTokenizer()
tokenizer_pt.fit_on_texts(data_splitted)
print("Word Indices (PyTorch):", tokenizer_pt.word_index)

vocab_size_pt = len(tokenizer_pt.word_index) + 1
print("Vocab Size (PyTorch):", vocab_size_pt)

sequences_pt = tokenizer_pt.texts_to_sequences(data_splitted)
print("Sequences (PyTorch):", sequences_pt, type(sequences_pt), len(sequences_pt))

X_pt = []
y_pt = []

# here from the sequences we are creating x pt which consist of the words excluding the last words and predicting the last waord as y
# and making x and y seperate
for seq in sequences_pt:
    X_pt.append(seq[:-1])
    y_pt.append(seq)

print("X_pt=", X_pt, "y_pt=", y_pt, type(X_pt), type(y_pt))




# now doing padding in the sequences
# here we will be finding the max length sentence and with that length will be adding the padding in the other sentences where the length is lesser than max

# Padding sequences
maxlen_pt = max([len(sequence) for sequence in X_pt])
print("Maxlen (PyTorch):", maxlen_pt)

# Keras pad_sequences defaults to 'pre' padding with 0. Maxlen is +1 for input X
# here also providing 2 conditions which can be change to by pre and post padding depenting of the user
def pad_sequences_pt(sequences, maxlen, padding='pre', value=0):
    padded_sequences = []
    for seq in sequences:
        if len(seq) >= maxlen:
            padded_sequences.append(seq[-maxlen:]) # Truncate if too long (Keras default is 'pre')
        else:
            pad_length = maxlen - len(seq)
            if padding == 'pre':
                padded_sequences.append([value] * pad_length + seq)
            elif padding == 'post':
                padded_sequences.append(seq + [value] * pad_length)
    return np.array(padded_sequences)



X_padded_pt = pad_sequences_pt(X_pt, maxlen=maxlen_pt + 1, padding='pre')
y_padded_pt = pad_sequences_pt(y_pt, maxlen=maxlen_pt + 1, padding='pre')

print("X_padded_pt:", X_padded_pt, type(X_padded_pt), X_padded_pt.shape)
print("y_padded_pt:", y_padded_pt, type(y_padded_pt), y_padded_pt.shape)

# Convert to PyTorch tensors
X_tensor = torch.tensor(X_padded_pt, dtype=torch.long)

# y needs to be one-hot encoded for categorical_crossentropy, shape (batch, seq_len, vocab_size)
y_one_hot = F.one_hot(torch.tensor(y_padded_pt, dtype=torch.long), num_classes=vocab_size_pt).float()

print("X_tensor shape:", X_tensor.shape)
print("y_one_hot shape:", y_one_hot.shape)
# print("y_one_hot shape:", y_one_hot)


# model defining

class PyTorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super(PyTorchRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True) # batch_first=True for (batch, seq, feature)
        self.dense = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, _ = self.rnn(embedded) # _ is the hidden state, not used here for simple RNN
        # Apply dense layer to each time step's output
        # translates that "memory" into a score for every word in your dictionary so you can pick the winner.
        output = self.dense(output)
        return output

# Word index → Embedding -> RNN -> Dense -> Next word prediction

embedding_dim = 10
hidden_size = 50
model_pt = PyTorchRNN(vocab_size_pt, embedding_dim, hidden_size)
print(model_pt)


# training

optimizer_pt = optim.RMSprop(model_pt.parameters())
# CrossEntropyLoss expects (N, C, ...) for input and (N, ...) for target
# Our output is (batch, seq_len, vocab_size), target is (batch, seq_len, vocab_size) one-hot
# We need to reshape and then apply F.log_softmax and NLLLoss, or flatten and use CrossEntropyLoss

# For simplicity, we will flatten and use CrossEntropyLoss which combines LogSoftmax and NLLLoss
# Input to CrossEntropyLoss should be (N, C) and target (N) where N is total number of elements

def train_pytorch_model(model, X_train, y_train_one_hot, epochs, optimizer):
    model.train() # Set the model to training mode
    criterion = nn.CrossEntropyLoss() # This implicitly applies LogSoftmax and then NLLLoss

    # Reshape y_train_one_hot to (batch_size * seq_len, vocab_size)
    # And get the target indices from the one-hot encoding for CrossEntropyLoss
    y_target_indices = torch.argmax(y_train_one_hot, dim=-1) # (batch_size, seq_len)

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass
        outputs = model(X_train) # (batch_size, seq_len, vocab_size)

        # Reshape outputs for CrossEntropyLoss: (batch_size * seq_len, vocab_size)
        outputs_flat = outputs.view(-1, vocab_size_pt)
        # Reshape target indices: (batch_size * seq_len)
        targets_flat = y_target_indices.view(-1)

        loss = criterion(outputs_flat, targets_flat)
        loss.backward() # Backward pass
        optimizer.step() # Optimize

        if (epoch + 1) % 50 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

epochs_pt = 200
train_pytorch_model(model_pt, X_tensor, y_one_hot, epochs_pt, optimizer_pt)

print("\n--- PyTorch Model Training Complete ---")


# Probability Calculation Function

def prob_of_input_sentence_pt(model, tokenizer, sentence, vocab_size):
    model.eval() # Set the model to evaluation model
    with torch.no_grad(): # Disable gradient calculations
        print("Input Sentence:", sentence)

        # Tokenize the sentence using the custom tokenizer
        encoded_list = tokenizer.texts_to_sequences([sentence])[0]

        # Keras `pad_sequences` behavior: inserts 0 at the beginning if input is shorter than maxlen.
        # Our X was padded with +1, so we need to match that.
        # The original Keras code inserted a 0 at the beginning manually, then padded.
        # We need to consider how the original maxlen was defined (maxlen_pt is for X_pt, which is already `seq[:-1]`).
        # The `X_tensor` was padded to `maxlen_pt + 1`.
        # So, the input to the model should be of `maxlen_pt + 1`.

        # First, ensure the '0' prefix for empty/short sequences as in Keras example
        # Original Keras code: encoded.insert(0,0) - this inserts 0 irrespective of actual length, then pad_sequences
        # So if `sentence` tokenizes to `[1, 2, 3]`, encoded becomes `[0, 1, 2, 3]` and then padded.

        input_seq_for_prob = [0] + encoded_list # Prepend 0 as in original Keras

        # Pad to the length the model expects, which is maxlen_pt + 1
        # Use our custom pad_sequences_pt with 'pre' padding
        padded_input = pad_sequences_pt([input_seq_for_prob], maxlen=maxlen_pt + 1, padding='pre')[0]

        encoded_tensor = torch.tensor(padded_input, dtype=torch.long).unsqueeze(0) # Add batch dimension
        print("Encoded (PyTorch):", encoded_tensor, encoded_tensor.shape)

        # Get model predictions (logits)
        logits = model(encoded_tensor) # (1, seq_len, vocab_size)

        # Apply softmax to get probabilities for each token at each step
        prob = F.softmax(logits, dim=-1) # (1, seq_len, vocab_size)
        print("Prob (PyTorch):", prob, prob.shape)

        probability = 1.0

        # Iterate through the sequence to calculate the joint probability
        # The Keras code calculated P(word_i | word_0...word_{i-1})
        # and multiplied these probabilities. This corresponds to the probability
        # of the actual next word in the sequence, given the previous ones.

        # The original Keras code iterates from i=0 to prob.shape[1]-1 (length-1)
        # and uses encoded[0, i+1] as the target word.

        # So for an input sequence `s_0, s_1, ..., s_N`, the model predicts `p_0, p_1, ..., p_N`
        # `p_0` is prob of `s_1` given `s_0`
        # `p_1` is prob of `s_2` given `s_0, s_1` and so on.
        # The Keras `encoded[0, i+1]` is the actual next word at position `i+1`.

        for i in range(prob.shape[1] - 1):
            # prob[0, i, :] gives the probability distribution for the next word at step i
            # encoded_tensor[0, i+1] is the actual next word index in the input sequence

            # Ensure the index is within bounds of vocab_size
            target_word_index = int(encoded_tensor[0, i+1].item())
            if target_word_index >= vocab_size:
                # This case might happen if token not in vocab, or it's a padding zero.
                # If it's a padding zero, its probability should be 1 if it's the target
                # Or we skip it if it's not a real word. Keras handles padding '0' gracefully.
                # We will assume 0 (PAD) has a probability of 1 if it's the target (for padding effect)
                # But if a word from the sentence itself is not in vocab, its prob is 0.
                # For this specific Keras example, all words are in vocab.
                # If target_word_index is 0 (PAD), we treat it as 1 for simplicity of padding effect.
                if target_word_index == 0:
                    word_prob = 1.0 # Or maybe skip. Original Keras would try to predict prob of 0.
                                    # For padding, it usually means we're done with the sentence.
                                    # The original Keras prob was calculated for 0 as well.
                else:
                    word_prob = 0.0 # Word not in vocab
            else:
                word_prob = prob[0, i, target_word_index].item()
            probability *= word_prob

        print("Probability of Sentence ", "\"" + sentence + "\"", "is:", probability)


print("-------------------Probability of Input Sentence (PyTorch)------------------------------")
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Rudra Went", vocab_size_pt)
prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Jill Went", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "Jack and Rudra Went", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and Jill Went up", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "went up the hill", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack and Jill Went up the hill .", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fetch a pail", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "to fetch a pail of water", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "fell down and broke", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "jack fell down and broke his crown", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "g after", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came tumbling after .", vocab_size_pt)
# prob_of_input_sentence_pt(model_pt, tokenizer_pt, "and jill came walkings after .", vocab_size_pt)
print("-------------------------------------------------------------------------------")

<>:17: SyntaxWarning: invalid escape sequence '\]'
<>:17: SyntaxWarning: invalid escape sequence '\]'
/tmp/ipython-input-2046748596.py:17: SyntaxWarning: invalid escape sequence '\]'
  def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\]^_`{|}~'):


Data:  Jack and Jill went up the hill .
 To fetch a pail of water .
 Jack fell down and broke his crown .
 And Jill came tumbling after . <class 'str'>
Data_Splitted: [' Jack and Jill went up the hill .', ' To fetch a pail of water .', ' Jack fell down and broke his crown .', ' And Jill came tumbling after .'] <class 'list'>
Word Indices (PyTorch): {'jack': 1, 'and': 2, 'jill': 3, 'went': 4, 'up': 5, 'the': 6, 'hill': 7, '.': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22}
Vocab Size (PyTorch): 23
Sequences (PyTorch): [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'list'> 4
X_pt= [[1, 2, 3, 4, 5, 6, 7], [9, 10, 11, 12, 13, 14], [1, 15, 16, 2, 17, 18, 19], [2, 3, 20, 21, 22]] y_pt= [[1, 2, 3, 4, 5, 6, 7, 8], [9, 10, 11, 12, 13, 14, 8], [1, 15, 16, 2, 17, 18, 19, 8], [2, 3, 20, 21, 22, 8]] <class 'l

so in the above if we just find the sentnence witht the new word in that then the : when it encounters a word which is not presnet in the vocab then it will just drop the word from that and hence the padding is added in the front of the sentence:


"Jack and Rudra Went"
→ lowercased: "jack and rudra went"
→ words: ["jack", "and", "rudra", "went"]

"jack"  → 1
"and"   → 2
"rudra" → ❌ NOT IN VOCAB → DROPPED
"went"  → 4


[1, 2, 4]
[0, 1, 2, 4]
[0, 0, 0, 0, 0, 1, 2, 4]
[0, 0, 0, 0, 0, 1, 2, 4]
[PAD, PAD, PAD, PAD, PAD, jack, and, went]


jack → and → jill → went → up → the → hill → .
jack → and → went

That sequence is weird / unlikely according to training:
After "and", the model strongly expects "jill", not "went".
So P(went | jack and) is very small.
And since you multiply probabilities across steps, the final joint probability becomes tiny:

Jack and Rudra Went  → ~1.8e-10
Jack and Jill Went   → ~1.5e-05


In [ ]:
# trying in generating the sequence of the sentence: by fetching the higest next word probablity in the sequence: known as sampling in the sequence diagram

In [ ]:
# NEW++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++

# The below code is for replacing the unknown word with UNK
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np



data = """ Jack and Jill went up the hill .
To fetch a pail of water .
Jack fell down and broke his crown .
And Jill came tumbling after ."""

data_splitted = data.split('\n')




class PyTorchTokenizer:
    def __init__(self, filters='!"#$%&()*+,-/:;<=>?@[\\]^_`{|}~'):
        self.word_index = {"PAD": 0, "UNK": 1}
        self.index_word = {0: "PAD", 1: "UNK"}
        self.filters = filters
        self.word_count = {}

    def fit_on_texts(self, texts):
        current_index = 2   # Start after PAD & UNK

        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()

            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1
                self.word_count[word] = self.word_count.get(word, 0) + 1

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            cleaned_text = text.lower()
            for char in self.filters:
                cleaned_text = cleaned_text.replace(char, '')
            words = cleaned_text.split()

            # 🔥 KEY CHANGE: Use UNK for unseen words
            seq = [self.word_index.get(word, self.word_index["UNK"]) for word in words]
            sequences.append(seq)

        return sequences


tokenizer = PyTorchTokenizer()
tokenizer.fit_on_texts(data_splitted)

vocab_size = len(tokenizer.word_index)
print("Vocab Size:", vocab_size)
print("Word Index:", tokenizer.word_index)


sequences = tokenizer.texts_to_sequences(data_splitted)

X = []
y = []

for seq in sequences:
    X.append(seq[:-1])
    y.append(seq)

maxlen = max(len(seq) for seq in X)

def pad_sequences(sequences, maxlen, padding='pre'):
    padded = []
    for seq in sequences:
        if len(seq) < maxlen:
            pad_len = maxlen - len(seq)
            if padding == 'pre':
                padded.append([0]*pad_len + seq)
            else:
                padded.append(seq + [0]*pad_len)
        else:
            padded.append(seq[-maxlen:])
    return np.array(padded)

X_padded = pad_sequences(X, maxlen=maxlen+1)
y_padded = pad_sequences(y, maxlen=maxlen+1)

X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y_padded, dtype=torch.long)




class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=10, hidden_size=50):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        output, _ = self.rnn(x)
        output = self.fc(output)
        return output


model = SimpleRNN(vocab_size)
criterion = nn.CrossEntropyLoss()
optimizer = optim.RMSprop(model.parameters())



epochs = 300
model.train()

for epoch in range(epochs):

    optimizer.zero_grad()

    outputs = model(X_tensor)  # (batch, seq_len, vocab)

    outputs = outputs.view(-1, vocab_size)
    targets = y_tensor.view(-1)

    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

print("\nTraining Complete\n")




def sentence_probability(model, tokenizer, sentence):
    model.eval()
    with torch.no_grad():

        encoded = tokenizer.texts_to_sequences([sentence])[0]

        # Add PAD at beginning
        encoded = [0] + encoded

        padded = pad_sequences([encoded], maxlen=maxlen+1)[0]
        input_tensor = torch.tensor(padded, dtype=torch.long).unsqueeze(0)

        logits = model(input_tensor)
        probs = F.softmax(logits, dim=-1)

        probability = 1.0

        for i in range(probs.shape[1] - 1):
            next_word = input_tensor[0, i+1].item()
            probability *= probs[0, i, next_word].item()

        print(f"Sentence: \"{sentence}\"")
        print("Probability:", probability)
        print()




sentence_probability(model, tokenizer, "Jack and Jill went")
sentence_probability(model, tokenizer, "Jack and Rudra went")

Vocab Size: 24
Word Index: {'PAD': 0, 'UNK': 1, 'jack': 2, 'and': 3, 'jill': 4, 'went': 5, 'up': 6, 'the': 7, 'hill': 8, '.': 9, 'to': 10, 'fetch': 11, 'a': 12, 'pail': 13, 'of': 14, 'water': 15, 'fell': 16, 'down': 17, 'broke': 18, 'his': 19, 'crown': 20, 'came': 21, 'tumbling': 22, 'after': 23}
Epoch 50/300, Loss: 0.1827
Epoch 100/300, Loss: 0.1858
Epoch 150/300, Loss: 0.1774
Epoch 200/300, Loss: 0.1841
Epoch 250/300, Loss: 0.1791
Epoch 300/300, Loss: 0.1770

Training Complete

Sentence: "Jack and Jill went"
Probability: 8.342773090829103e-06

Sentence: "Jack and Rudra went"
Probability: 1.6214552710171865e-11



In [ ]:
# now below is the code for dynamic vocab adding when a new word is detected

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np



data = """ Jack and Jill went up the hill .
To fetch a pail of water .
Jack fell down and broke his crown .
And Jill came tumbling after ."""

data_splitted = data.split('\n')




class DynamicTokenizer:
    def __init__(self):
        self.word_index = {"PAD": 0, "UNK": 1}
        self.index_word = {0: "PAD", 1: "UNK"}

    def fit_on_texts(self, texts):
        current_index = 2
        for text in texts:
            words = text.lower().replace(".", "").split()
            for word in words:
                if word not in self.word_index:
                    self.word_index[word] = current_index
                    self.index_word[current_index] = word
                    current_index += 1

    def encode_with_unk(self, text):
        words = text.lower().replace(".", "").split()
        return [self.word_index.get(w, 1) for w in words]

    def add_new_words(self, text):
        words = text.lower().replace(".", "").split()
        new_words = []
        for w in words:
            if w not in self.word_index:
                new_index = len(self.word_index)
                self.word_index[w] = new_index
                self.index_word[new_index] = w
                new_words.append(w)
        return new_words


tokenizer = DynamicTokenizer()
tokenizer.fit_on_texts(data_splitted)

vocab_size = len(tokenizer.word_index)
print("Initial Vocab:", tokenizer.word_index)




class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_size=64):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        out = self.fc(out)
        return out


model = SimpleRNN(vocab_size)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()



def expand_model(model, new_vocab_size):
    old_vocab_size, emb_dim = model.embedding.weight.shape
    hidden_size = model.fc.in_features

    # ---- Expand Embedding ----
    new_embedding = nn.Embedding(new_vocab_size, emb_dim)
    new_embedding.weight.data[:old_vocab_size] = model.embedding.weight.data
    nn.init.normal_(new_embedding.weight.data[old_vocab_size:])

    # ---- Expand Output Layer ----
    new_fc = nn.Linear(hidden_size, new_vocab_size)
    new_fc.weight.data[:old_vocab_size] = model.fc.weight.data
    new_fc.bias.data[:old_vocab_size] = model.fc.bias.data

    model.embedding = new_embedding
    model.fc = new_fc

    return model



def train_on_sentence(sentence, epochs=100):
    global model, optimizer

    # Encode using UNK
    encoded = tokenizer.encode_with_unk(sentence)

    if len(encoded) < 2:
        return

    X = torch.tensor([encoded[:-1]], dtype=torch.long)
    y = torch.tensor([encoded[1:]], dtype=torch.long)

    for _ in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output.view(-1, len(tokenizer.word_index)),
                         y.view(-1))
        loss.backward()
        optimizer.step()



def   (sentence):
    global model, optimizer

    print("\nINPUT:", sentence)

    # ---- First pass: treat unknown as UNK ----
    encoded = tokenizer.encode_with_unk(sentence)
    print("Encoded (with UNK):", encoded)

    # Train once
    train_on_sentence(sentence, epochs=50)

    # ---- Now Add New Words ----
    new_words = tokenizer.add_new_words(sentence)

    if new_words:
        print("New words added:", new_words)

        new_vocab_size = len(tokenizer.word_index)

        # Expand model
        model = expand_model(model, new_vocab_size)

        # Reinitialize optimizer for new parameters
        optimizer = optim.Adam(model.parameters(), lr=0.01)

    print("Updated Vocab:", tokenizer.word_index)




dynamic_process("Jack and Rudra went")
dynamic_process("Jack and Rudra went")

Initial Vocab: {'PAD': 0, 'UNK': 1, 'jack': 2, 'and': 3, 'jill': 4, 'went': 5, 'up': 6, 'the': 7, 'hill': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22}

INPUT: Jack and Rudra went
Encoded (with UNK): [2, 3, 1, 5]
New words added: ['rudra']
Updated Vocab: {'PAD': 0, 'UNK': 1, 'jack': 2, 'and': 3, 'jill': 4, 'went': 5, 'up': 6, 'the': 7, 'hill': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22, 'rudra': 23}

INPUT: Jack and Rudra went
Encoded (with UNK): [2, 3, 23, 5]
Updated Vocab: {'PAD': 0, 'UNK': 1, 'jack': 2, 'and': 3, 'jill': 4, 'went': 5, 'up': 6, 'the': 7, 'hill': 8, 'to': 9, 'fetch': 10, 'a': 11, 'pail': 12, 'of': 13, 'water': 14, 'fell': 15, 'down': 16, 'broke': 17, 'his': 18, 'crown': 19, 'came': 20, 'tumbling': 21, 'after': 22, 'rudra